In [1]:
using QuantumOptics
using IonSim
using Pkg;
Pkg.add("DSP")
Pkg.add("LaTeXStrings")
using DSP: periodogram
using LaTeXStrings
import PyPlot
const plt = PyPlot

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


: 

In [ ]:
# set some plot configs
plt.matplotlib.rc("xtick", top=false)
plt.matplotlib.rc("ytick", right=false, left=false)
plt.matplotlib.rc("axes", labelsize=20, titlesize=20, grid=true)
plt.matplotlib.rc("axes", linewidth=2)
plt.matplotlib.rc("grid", alpha=0.25, linestyle="--")
plt.matplotlib.rc("font", family="Palatino", weight="medium")
plt.matplotlib.rc("figure", figsize=(8,4))
plt.matplotlib.rc("xtick.major", width=2)
plt.matplotlib.rc("ytick.major", width=2)

In [ ]:
# Setup system
ca = Ca40([("S1/2", -1/2, "S"), ("D5/2", -1/2, "D")])
laser = Laser()
νᵣ = √14 * 1e6  # radial trap frequency
νₐ = 1.5e6  # axial trap frequency
chain = LinearChain(
        ions=[ca],
        comfrequencies=(x=νᵣ, y=νᵣ, z=νₐ), 
        selectedmodes=(x=[1], z=[1]),
        N = 2
    )
chamber = Chamber(iontrap=chain, B=4e-4, Bhat=ẑ, δB=0, lasers=[laser])
# set laser parameters
wavelength_from_transition!(laser, ca, ("S", "D"), chamber)
polarization!(laser, ẑ)
wavevector!(laser, (x̂ + ẑ)/√2)
# set carrier transition Rabi frequency to 500 kHz
intensity_from_rabifrequency!(1, 5e5, 1, ("S", "D"), chamber);

In [ ]:
axial = zmodes(chamber)[1]
radial = xmodes(chamber)[1]

ρᵢ_ion = dm(ca["S"])
ρᵢ_axial = thermalstate(axial, 0.5)
ρᵢ_radial = thermalstate(radial, 0.5)
# Set initial state to ρᵢ = |↓, n̄ᵣ=0.5, n̄ₐ=0.5⟩
ρᵢ = ρᵢ_ion ⊗ ρᵢ_radial ⊗ ρᵢ_axial;  

In [ ]:
tspan = 0:10:400
fout(t, ρ) = real(expect(ionprojector(chamber, "D"), ρ))
J = (-dm(ca["S"]) + dm(ca["D"])) ⊗ one(radial) ⊗ one(axial)  # Collapse operator
γ = 1e4 * 1e-6 

# Scan laser detuning
Δlist = range(νₐ, stop=2νᵣ + 1e5, length=3)
exclist = []
for Δ in Δlist
    detuning!(laser, Δ)
    h = hamiltonian(chamber, timescale=1e-6, rwa_cutoff=1.5e6, lamb_dicke_order=2)
    _, sol = timeevolution.master_dynamic(tspan, ρᵢ, (t, ρ) -> (h(t, ρ), [J], [J], [γ]), fout=fout)
    push!(exclist, real(sol[end]))
end 